In [12]:
cd /Users/karolinegriesbach/Documents/Innkeepr/Git/evaluation-and-execution-scripts/

In [13]:
import logging
import pandas as pd
import numpy as np
import awswrangler as wr
from matplotlib import pyplot as plt
import seaborn as sns
from general_functions.return_workspace_ids import return_workspace_ids
from general_functions.constants import return_api_url
from general_functions.call_api_with_account_id import call_api_with_accountId, send_to_innkeepr_api_paginated

In [14]:
customer = "More"
start_date = "20260212"
end_date = "20260212"
date_range = pd.date_range(start=start_date, end=end_date, freq="D").strftime("%Y%m%d").tolist()
url = return_api_url()
print(f"url = {url}")
account_id = return_workspace_ids()
account_id = [acc["id"] for acc in account_id if acc["name"] == customer]
account_id = account_id[0]

In [15]:
data_file_path = f"DataChecks/targeting_history_meta_conversion_update/data/targeting_history_{customer}_{start_date}_{end_date}_test.csv"
try:
    df = pd.read_csv(data_file_path)
except FileNotFoundError:
    print("File not found, creating new DataFrame.")
    df = pd.DataFrame()
    for date in date_range:
        try:
            print(f"Reading data for {date}")
            path = f"s3://innkeepr-development/targeting.history/{date}/meta_conversion_update_2026-02-1209:39:11.825851.parquet"
            print(path)
            temp = wr.s3.read_parquet(path)
        except wr.exceptions.NoFilesFound:
            print(f". No data for {date}")
            continue
        temp["bucket_date"] = date
        df = pd.concat([df, temp])
    df.to_csv(data_file_path, index=False)
df

In [16]:
df.groupby("created")["anonymousId"].nunique()

In [17]:
df.groupby("session.date")["anonymousId"].nunique()

In [24]:
df[["final_adjusted_revenue", "properties.revenue"]] = df[["final_adjusted_revenue", "properties.revenue"]].apply(pd.to_numeric)
df_conversion_sum = df.drop_duplicates(subset=["session"])
print(df_conversion_sum[["final_adjusted_revenue", "properties.revenue"]].sum())
df_conversion_sum["final_adjusted_revenue"].describe()

In [22]:
df_conversion_sum["properties.revenue"].describe()

In [26]:
df_conversion_sum["final_multiplier"] = df_conversion_sum["final_multiplier"].astype("float")
df_conversion_sum["final_multiplier"].describe()

In [28]:
df_conversion_sum["final_multiplier"].hist()
plt.title("Historgram of final multiplier")

In [34]:
#relates_to.adset.name
df_conversion_sum["label"] = df_conversion_sum["relates_to.adset.name"].str[0:20]
df_conversion_sum.groupby("relates_to.adset.name")["final_multiplier"].describe()


In [38]:
fig, ax = plt.subplots(figsize=(16, 8))
sns.boxplot(data=df_conversion_sum, x="relates_to.adset.name", y="final_multiplier", ax=ax)
plt.grid(True)
plt.xticks(rotation=90)
plt.title("Boxplot of final multiplier per adset")
plt.tight_layout()
plt.show()

In [41]:
fig, ax = plt.subplots(figsize=(16, 8))
df_conversion_sum["diff"] = df_conversion_sum["final_adjusted_revenue"] - df_conversion_sum["properties.revenue"]
sns.boxplot(data=df_conversion_sum, x="relates_to.adset.name", y="diff", ax=ax)
plt.grid(True)
plt.xticks(rotation=90)
plt.title("Boxplot of diff per adset")
plt.tight_layout()
plt.show()

# Compare with old one

In [46]:
data_file_path_prod = f"DataChecks/targeting_history_meta_conversion_update/data/targeting_history_{customer}_{start_date}_{end_date}.csv"
df_prod = pd.read_csv(data_file_path_prod)
df_prod_conv = df_prod.drop_duplicates(subset=["session"])
df_prod_conv[["final_adjusted_revenue", "properties.revenue"]] = df_prod_conv[["final_adjusted_revenue", "properties.revenue"]].apply(pd.to_numeric)
df_prod_conv = pd.merge(df_prod_conv, df_conversion_sum[["relates_to.adset.name", "session"]], on="session")


In [63]:
df_conversion_sum[df_conversion_sum["relates_to.adset.name"]!="no_adset"]["relates_to.adset.name"].count(), df_conversion_sum[df_conversion_sum["relates_to.adset.name"]=="no_adset"]["relates_to.adset.name"].count(), 

In [61]:
from heapq import merge


use_columns_revenue: list[str] = ["session","relates_to.adset.name", "properties.revenue"]
use_columns_adj_revenue: list[str] = ["session","relates_to.adset.name", "final_adjusted_revenue"]
df_prod_revenue = df_prod_conv[use_columns_revenue].rename(columns={"properties.revenue": "revenue"})
df_prod_revenue["label"] = "current revenue"
df_prod_adj_revenue = df_prod_conv[use_columns_adj_revenue].rename(columns={"final_adjusted_revenue": "revenue"})
df_prod_adj_revenue["label"] = "current adjusted revenue"
df_test_revenue = df_conversion_sum[use_columns_revenue].rename(columns={"properties.revenue": "revenue"})
df_test_revenue["label"] = "test revenue"
df_test_adj_revenue = df_conversion_sum[use_columns_adj_revenue].rename(columns={"final_adjusted_revenue": "revenue"})
df_test_adj_revenue["label"] = "test adjusted revenue"

merged = pd.concat([df_prod_revenue, df_prod_adj_revenue, df_test_revenue, df_test_adj_revenue])
merged_stats = merged.groupby(by=["label", "relates_to.adset.name"])["revenue"].describe()
merged_stats = merged_stats.reset_index()
merged_stats = merged_stats.sort_values(by=["relates_to.adset.name", "label"])
merged_stats


In [57]:
sns.boxplot(data=merged, x="relates_to.adset.name", y="revenue", hue="label")
plt.grid(True)
plt.xticks(rotation=90)
plt.title("Boxplot of revenues per adset")
plt.tight_layout()
plt.show()

In [58]:
sns.boxplot(data=merged, x="relates_to.adset.name", y="revenue", hue="label")
plt.grid(True)
plt.xticks(rotation=90)
plt.title("Boxplot of revenues per adset")
plt.ylim(0, 200)
plt.tight_layout()
plt.show()

In [47]:
prod_adj_revenue_by_adset = df_prod_conv.groupby("relates_to.adset.name")["final_adjusted_revenue"].describe()
prod_adj_revenue_by_adset = pd.DataFrame(prod_adj_revenue_by_adset)
prod_adj_revenue_by_adset["label"] = "current adjusted revenue"
prod_revenue_by_adset = df_prod_conv.groupby("relates_to.adset.name")["properties.revenue"].describe()
prod_revenue_by_adset = pd.DataFrame(prod_revenue_by_adset)
prod_revenue_by_adset["label"] = "current revenue"
test_adj_revenue_by_adset = df_conversion_sum.groupby("relates_to.adset.name")["final_adjusted_revenue"].describe()
test_adj_revenue_by_adset = pd.DataFrame(test_adj_revenue_by_adset)
test_adj_revenue_by_adset["label"] = "test adjusted revenue"
test_revenue_by_adset = df_conversion_sum.groupby("relates_to.adset.name")["properties.revenue"].describe()
test_revenue_by_adset = pd.DataFrame(test_revenue_by_adset)
test_revenue_by_adset["label"] = "test revenue"
stats = pd.concat([prod_adj_revenue_by_adset, prod_revenue_by_adset, test_adj_revenue_by_adset, test_revenue_by_adset])
stats = stats.fillna(0)
stats

In [51]:
fig, ax = plt.subplots(figsize=(16, 8))
sns.barplot(data=stats, x="relates_to.adset.name", y="mean", hue="label", ax=ax)
ax.set_ylabel("revenue")
ax.set_title("Mean final revenue per adset (with std error bars)")
plt.xticks(rotation=90)
plt.grid(True, axis="y")
plt.tight_layout()
plt.show()